 01_results_analysis_vllm — normalize vLLM benchmark outputs, compute metrics, export processed tables

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_RESULTS_DIR = ROOT / "results" / "raw"
PROCESSED_DIR = ROOT / "results" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]

print("ROOT:", ROOT)
print("RAW_RESULTS_DIR:", RAW_RESULTS_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


ROOT: /home/stephane/resource-efficiency-benchmark
RAW_RESULTS_DIR: /home/stephane/resource-efficiency-benchmark/results/raw
PROCESSED_DIR: /home/stephane/resource-efficiency-benchmark/results/processed


 Discover vLLM runs and load raw files

In [2]:
def discover_runs(results_dir: Path) -> pd.DataFrame:
    rows = []

    if not results_dir.exists():
        return pd.DataFrame(columns=["model_alias_dir", "run_id", "run_dir"])

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue

        for run_dir in sorted(model_dir.iterdir()):
            if not run_dir.is_dir():
                continue

            rows.append({
                "model_alias_dir": model_dir.name,
                "run_id": run_dir.name,
                "run_dir": str(run_dir),
            })

    return pd.DataFrame(rows)


def safe_read_csv(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"FAILED reading {path}: {e}")
        return pd.DataFrame()


def load_raw_outputs(results_dir: Path):
    runs_df = discover_runs(results_dir)

    summary_tables = []
    sample_tables = []

    for _, r in runs_df.iterrows():
        run_dir = Path(r["run_dir"])

        for ds in DATASETS:
            summary_path = run_dir / f"{ds}.csv"
            samples_path = run_dir / f"{ds}_samples.csv"

            if summary_path.exists():
                df = safe_read_csv(summary_path)
                if not df.empty:
                    df["model_alias_dir"] = r["model_alias_dir"]
                    df["run_id"] = r["run_id"]
                    df["dataset"] = ds
                    df["source_path"] = str(summary_path)
                    summary_tables.append(df)

            if samples_path.exists():
                df = safe_read_csv(samples_path)
                if not df.empty:
                    df["model_alias_dir"] = r["model_alias_dir"]
                    df["run_id"] = r["run_id"]
                    df["dataset"] = ds
                    df["source_path"] = str(samples_path)
                    sample_tables.append(df)

    raw_summary = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
    raw_samples = pd.concat(sample_tables, ignore_index=True) if sample_tables else pd.DataFrame()

    return runs_df, raw_summary, raw_samples


runs_df, raw_summary, raw_samples = load_raw_outputs(RAW_RESULTS_DIR)

print("runs_df:", runs_df.shape)
print("raw_summary:", raw_summary.shape)
print("raw_samples:", raw_samples.shape)


runs_df: (5, 3)
raw_summary: (20, 34)
raw_samples: (15500, 35)


 Quick inspection

In [3]:
display(runs_df.head())
display(raw_summary.head())
display(raw_samples.head())


,model_alias_dir,run_id,run_dir
0,llama3_2_1b,2026-04-07_13-35-14,/home/stephane/resource-efficiency-benchmark/r...
1,ministral3_3b_awq,2026-04-07_13-56-00,/home/stephane/resource-efficiency-benchmark/r...
2,phi4mini_gptq_4bit,2026-04-07_16-29-17,/home/stephane/resource-efficiency-benchmark/r...
3,qwen2_5_1_5b,2026-04-07_14-29-28,/home/stephane/resource-efficiency-benchmark/r...
4,qwen3_4b_awq,2026-04-07_14-50-20,/home/stephane/resource-efficiency-benchmark/r...


,Model alias,Repo id,Served model id,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,GPU percent avg mean,GPU percent avg sd,Power W avg mean,Power W avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,GPU percent peak,VRAM used MB peak,Power W peak,model_alias_dir,run_id,dataset,source_path
0,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.44,0.0,68.2622,0.3665,122.6156,0.7112,18.4056,0.1139,6.0300,0.2637,68.5707,0.6288,8.2657,9.4386,58.1622,0.6932,5.6841,0.0316,5.6841,0.0316,39.2,9376.24,896.96,71.500,22.0,5436.0,61.44,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...
1,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.24,0.0,68.1985,0.1777,122.4676,0.4120,18.4337,0.0793,5.9284,0.1299,67.6965,0.2944,2.5020,6.6964,58.2584,0.0683,5.6900,0.0191,5.6900,0.0191,43.4,9272.16,898.32,68.625,22.0,5400.0,60.20,llama3_2_1b,2026-04-07_13-35-14,hellaswag,/home/stephane/resource-efficiency-benchmark/r...
2,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.06,0.0,67.0925,0.2472,138.3586,0.3620,53.9784,0.0768,6.8413,0.0165,71.7561,0.2473,44.0157,15.2163,103.1411,1.4601,23.6684,0.0713,5.2364,0.0158,23.6,9387.93,899.57,73.250,100.0,5370.0,128.19,llama3_2_1b,2026-04-07_13-35-14,gsm8k,/home/stephane/resource-efficiency-benchmark/r...
3,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,NaN,NaN,73.7217,5.8875,471.5200,54.9763,109.4824,4.3976,7.9899,0.2048,74.9008,0.1878,66.0237,30.1451,158.7051,4.3896,83.8610,4.9172,1.9239,0.1934,22.2,9301.80,899.69,76.250,100.0,5158.0,187.04,llama3_2_1b,2026-04-07_13-35-14,dolly,/home/stephane/resource-efficiency-benchmark/r...
4,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,0.64,0.0,82.6019,0.9471,137.8528,0.9156,18.1055,0.0572,5.9592,0.0430,70.0784,0.2781,28.6720,8.2209,69.6547,1.8607,5.2170,0.0256,5.2170,0.0256,23.9,10783.76,907.05,73.250,100.0,5396.0,102.96,ministral3_3b_awq,2026-04-07_13-56-00,mmlu,/home/stephane/resource-efficiency-benchmark/r...


,repo_id,served_model_id,model_alias,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_util_percent_avg,gpu_util_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,metric_points_json
0,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C,1,79.3070,133.0174,18.6184,4.6000,6.3,9370.37,895.34,68.6250,68.625,3.0,3.0,5254.0,47.9233,48.40,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN
1,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,67.7943,122.0332,18.4369,6.7000,7.3,9375.46,895.34,68.6250,68.625,3.0,3.0,5254.0,49.9367,50.54,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN
2,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,67.4295,122.5596,18.1389,5.6000,7.3,9376.24,895.34,68.5833,68.625,3.0,3.0,5254.0,51.2400,52.44,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN
3,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,C,False,C,1,66.5789,121.0383,18.3623,6.0000,7.3,9353.32,895.34,68.5833,68.625,3.0,3.0,5254.0,52.9467,53.94,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN
4,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,C,False,C,1,67.9756,122.0342,18.4985,5.5667,7.3,9364.48,895.34,68.6667,68.750,3.0,3.0,5254.0,54.7267,55.12,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN


 Normalize vLLM summary files

In [4]:
SUMMARY_MAP = {
    "Model alias": "model",
    "Repo id": "repo_id",
    "Served model id": "served_model_id",
    "Accuracy mean": "accuracy",
    "Accuracy sd": "accuracy_sd",
    "TTFT ms mean": "ttft",
    "TTFT ms sd": "ttft_sd",
    "End-to-end latency ms mean": "latency",
    "End-to-end latency ms sd": "latency_sd",
    "Tokens/sec mean": "tps",
    "Tokens/sec sd": "tps_sd",
    "CPU percent avg mean": "cpu",
    "CPU percent avg sd": "cpu_sd",
    "CPU percent peak": "cpu_peak",
    "RAM used MB peak": "ram_used_mb_peak",
    "Process RAM MB peak": "ram",
    "Temp C avg mean": "temp_c_avg",
    "Temp C avg sd": "temp_c_avg_sd",
    "Temp C peak": "temp_c_peak",
    "GPU percent avg mean": "gpu",
    "GPU percent avg sd": "gpu_sd",
    "GPU percent peak": "gpu_peak",
    "VRAM used MB peak": "vram",
    "Power W avg mean": "power",
    "Power W avg sd": "power_sd",
    "Power W peak": "power_peak",
    "Sequential sustained tokens/sec mean": "sustained_tps",
    "Sequential sustained tokens/sec sd": "sustained_tps_sd",
    "Sequential sustained samples/sec mean": "sustained_sps",
    "Sequential sustained samples/sec sd": "sustained_sps_sd",
}

def normalize_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = []

    for _, r in df.iterrows():
        row = {
            "backend": "vllm",
            "run_id": r["run_id"],
            "dataset": r["dataset"],
            "model_alias_dir": r["model_alias_dir"],
            "source_path": r["source_path"],
        }

        working = dict(r)

        for raw_col, clean_col in SUMMARY_MAP.items():
            row[clean_col] = working.get(raw_col)

        if pd.isna(row.get("model")):
            row["model"] = r.get("model_alias_dir")

        out.append(row)

    summary = pd.DataFrame(out)

    numeric_cols = [
        "accuracy", "accuracy_sd",
        "ttft", "ttft_sd",
        "latency", "latency_sd",
        "tps", "tps_sd",
        "cpu", "cpu_sd", "cpu_peak",
        "ram_used_mb_peak", "ram",
        "temp_c_avg", "temp_c_avg_sd", "temp_c_peak",
        "gpu", "gpu_sd", "gpu_peak",
        "vram",
        "power", "power_sd", "power_peak",
        "sustained_tps", "sustained_tps_sd",
        "sustained_sps", "sustained_sps_sd",
    ]

    for col in numeric_cols:
        if col in summary.columns:
            summary[col] = pd.to_numeric(summary[col], errors="coerce")

    return summary


summary = normalize_summary(raw_summary)
summary.head()


,backend,run_id,dataset,model_alias_dir,source_path,model,repo_id,served_model_id,accuracy,accuracy_sd,ttft,ttft_sd,latency,latency_sd,tps,tps_sd,cpu,cpu_sd,cpu_peak,ram_used_mb_peak,ram,temp_c_avg,temp_c_avg_sd,temp_c_peak,gpu,gpu_sd,gpu_peak,vram,power,power_sd,power_peak,sustained_tps,sustained_tps_sd,sustained_sps,sustained_sps_sd
0,vllm,2026-04-07_13-35-14,mmlu,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.44,0.0,68.2622,0.3665,122.6156,0.7112,18.4056,0.1139,6.0300,0.2637,39.2,9376.24,896.96,68.5707,0.6288,71.500,8.2657,9.4386,22.0,5436.0,58.1622,0.6932,61.44,5.6841,0.0316,5.6841,0.0316
1,vllm,2026-04-07_13-35-14,hellaswag,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.24,0.0,68.1985,0.1777,122.4676,0.4120,18.4337,0.0793,5.9284,0.1299,43.4,9272.16,898.32,67.6965,0.2944,68.625,2.5020,6.6964,22.0,5400.0,58.2584,0.0683,60.20,5.6900,0.0191,5.6900,0.0191
2,vllm,2026-04-07_13-35-14,gsm8k,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.06,0.0,67.0925,0.2472,138.3586,0.3620,53.9784,0.0768,6.8413,0.0165,23.6,9387.93,899.57,71.7561,0.2473,73.250,44.0157,15.2163,100.0,5370.0,103.1411,1.4601,128.19,23.6684,0.0713,5.2364,0.0158
3,vllm,2026-04-07_13-35-14,dolly,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,NaN,NaN,73.7217,5.8875,471.5200,54.9763,109.4824,4.3976,7.9899,0.2048,22.2,9301.80,899.69,74.9008,0.1878,76.250,66.0237,30.1451,100.0,5158.0,158.7051,4.3896,187.04,83.8610,4.9172,1.9239,0.1934
4,vllm,2026-04-07_13-56-00,mmlu,ministral3_3b_awq,/home/stephane/resource-efficiency-benchmark/r...,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,0.64,0.0,82.6019,0.9471,137.8528,0.9156,18.1055,0.0572,5.9592,0.0430,23.9,10783.76,907.05,70.0784,0.2781,73.250,28.6720,8.2209,100.0,5396.0,69.6547,1.8607,102.96,5.2170,0.0256,5.2170,0.0256


 Normalize vLLM sample files

In [5]:
SAMPLE_RENAME_MAP = {
    "repo_id": "repo_id",
    "served_model_id": "served_model_id",
    "model_alias": "model",
    "dataset_name": "dataset_name",
    "run_idx": "run_idx",
    "sample_idx": "sample_idx",
    "prompt": "prompt",
    "question": "question",
    "choices": "choices",
    "instruction": "instruction",
    "context": "context",
    "gold_answer": "gold_answer",
    "parsed_prediction": "parsed_prediction",
    "correct": "correct",
    "raw_output": "raw_output",
    "output_tokens": "output_tokens",
    "ttft_ms": "ttft_ms",
    "end_to_end_latency_ms": "latency_ms",
    "tokens_per_sec": "tokens_per_sec",
    "cpu_percent_avg": "cpu_percent_avg",
    "cpu_percent_peak": "cpu_percent_peak",
    "ram_used_mb_peak": "ram_used_mb_peak",
    "process_ram_mb_peak": "process_ram_mb_peak",
    "temp_c_avg": "temp_c_avg",
    "temp_c_peak": "temp_c_peak",
    "gpu_util_percent_avg": "gpu_percent_avg",
    "gpu_util_percent_peak": "gpu_percent_peak",
    "vram_used_mb_peak": "vram_used_mb_peak",
    "power_w_avg": "power_w_avg",
    "power_w_peak": "power_w_peak",
}

def normalize_samples(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    keep_cols = [c for c in SAMPLE_RENAME_MAP if c in df.columns]
    base_cols = ["model_alias_dir", "run_id", "dataset", "source_path"]

    samples = df[keep_cols + base_cols].rename(columns=SAMPLE_RENAME_MAP).copy()
    samples["backend"] = "vllm"

    if "model" not in samples.columns:
        samples["model"] = samples["model_alias_dir"]

    numeric_cols = [
        "run_idx", "sample_idx", "output_tokens",
        "ttft_ms", "latency_ms", "tokens_per_sec",
        "cpu_percent_avg", "cpu_percent_peak",
        "ram_used_mb_peak", "process_ram_mb_peak",
        "temp_c_avg", "temp_c_peak",
        "gpu_percent_avg", "gpu_percent_peak",
        "vram_used_mb_peak",
        "power_w_avg", "power_w_peak",
    ]

    for col in numeric_cols:
        if col in samples.columns:
            samples[col] = pd.to_numeric(samples[col], errors="coerce")

    if "correct" in samples.columns:
        samples["correct"] = samples["correct"].map(
            lambda x: np.nan if pd.isna(x) else bool(x)
        )

    return samples


samples = normalize_samples(raw_samples)
samples.head()


,repo_id,served_model_id,model,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_percent_avg,gpu_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,backend
0,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C,1,79.3070,133.0174,18.6184,4.6000,6.3,9370.37,895.34,68.6250,68.625,3.0,3.0,5254.0,47.9233,48.40,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm
1,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,67.7943,122.0332,18.4369,6.7000,7.3,9375.46,895.34,68.6250,68.625,3.0,3.0,5254.0,49.9367,50.54,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm
2,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,67.4295,122.5596,18.1389,5.6000,7.3,9376.24,895.34,68.5833,68.625,3.0,3.0,5254.0,51.2400,52.44,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm
3,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,C,False,C,1,66.5789,121.0383,18.3623,6.0000,7.3,9353.32,895.34,68.5833,68.625,3.0,3.0,5254.0,52.9467,53.94,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm
4,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,C,False,C,1,67.9756,122.0342,18.4985,5.5667,7.3,9364.48,895.34,68.6667,68.750,3.0,3.0,5254.0,54.7267,55.12,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm


 Derived metrics on summary tables

In [6]:
def add_summary_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["latency_s"] = out["latency"] / 1000.0
    out["ttft_s"] = out["ttft"] / 1000.0
    out["ttft_share_of_latency"] = out["ttft"] / out["latency"]

    out["tps_per_ram"] = out["tps"] / out["ram"]
    out["tps_per_vram"] = out["tps"] / out["vram"]
    out["tps_per_watt"] = out["tps"] / out["power"]
    out["sustained_tps_per_vram"] = out["sustained_tps"] / out["vram"]

    out["accuracy_per_latency"] = out["accuracy"] / out["latency"]
    out["accuracy_per_ram"] = out["accuracy"] / out["ram"]
    out["accuracy_per_vram"] = out["accuracy"] / out["vram"]
    out["accuracy_per_watt"] = out["accuracy"] / out["power"]

    return out


summary = add_summary_metrics(summary)
summary.head()


,backend,run_id,dataset,model_alias_dir,source_path,model,repo_id,served_model_id,accuracy,accuracy_sd,ttft,ttft_sd,latency,latency_sd,tps,tps_sd,cpu,cpu_sd,cpu_peak,ram_used_mb_peak,ram,temp_c_avg,temp_c_avg_sd,temp_c_peak,gpu,gpu_sd,gpu_peak,vram,power,power_sd,power_peak,sustained_tps,sustained_tps_sd,sustained_sps,sustained_sps_sd,latency_s,ttft_s,ttft_share_of_latency,tps_per_ram,tps_per_vram,tps_per_watt,sustained_tps_per_vram,accuracy_per_latency,accuracy_per_ram,accuracy_per_vram,accuracy_per_watt
0,vllm,2026-04-07_13-35-14,mmlu,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.44,0.0,68.2622,0.3665,122.6156,0.7112,18.4056,0.1139,6.0300,0.2637,39.2,9376.24,896.96,68.5707,0.6288,71.500,8.2657,9.4386,22.0,5436.0,58.1622,0.6932,61.44,5.6841,0.0316,5.6841,0.0316,0.122616,0.068262,0.556717,0.020520,0.003386,0.316453,0.001046,0.003588,0.000491,0.000081,0.007565
1,vllm,2026-04-07_13-35-14,hellaswag,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.24,0.0,68.1985,0.1777,122.4676,0.4120,18.4337,0.0793,5.9284,0.1299,43.4,9272.16,898.32,67.6965,0.2944,68.625,2.5020,6.6964,22.0,5400.0,58.2584,0.0683,60.20,5.6900,0.0191,5.6900,0.0191,0.122468,0.068198,0.556870,0.020520,0.003414,0.316413,0.001054,0.001960,0.000267,0.000044,0.004120
2,vllm,2026-04-07_13-35-14,gsm8k,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.06,0.0,67.0925,0.2472,138.3586,0.3620,53.9784,0.0768,6.8413,0.0165,23.6,9387.93,899.57,71.7561,0.2473,73.250,44.0157,15.2163,100.0,5370.0,103.1411,1.4601,128.19,23.6684,0.0713,5.2364,0.0158,0.138359,0.067092,0.484917,0.060005,0.010052,0.523345,0.004408,0.000434,0.000067,0.000011,0.000582
3,vllm,2026-04-07_13-35-14,dolly,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,NaN,NaN,73.7217,5.8875,471.5200,54.9763,109.4824,4.3976,7.9899,0.2048,22.2,9301.80,899.69,74.9008,0.1878,76.250,66.0237,30.1451,100.0,5158.0,158.7051,4.3896,187.04,83.8610,4.9172,1.9239,0.1934,0.471520,0.073722,0.156349,0.121689,0.021226,0.689848,0.016258,NaN,NaN,NaN,NaN
4,vllm,2026-04-07_13-56-00,mmlu,ministral3_3b_awq,/home/stephane/resource-efficiency-benchmark/r...,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,0.64,0.0,82.6019,0.9471,137.8528,0.9156,18.1055,0.0572,5.9592,0.0430,23.9,10783.76,907.05,70.0784,0.2781,73.250,28.6720,8.2209,100.0,5396.0,69.6547,1.8607,102.96,5.2170,0.0256,5.2170,0.0256,0.137853,0.082602,0.599204,0.019961,0.003355,0.259932,0.000967,0.004643,0.000706,0.000119,0.009188


 Derived metrics on sample tables

In [7]:
def add_sample_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["latency_s"] = out["latency_ms"] / 1000.0
    out["ttft_s"] = out["ttft_ms"] / 1000.0
    out["ttft_share_of_latency"] = out["ttft_ms"] / out["latency_ms"]

    out["tokens_per_ram"] = out["tokens_per_sec"] / out["process_ram_mb_peak"]
    out["tokens_per_vram"] = out["tokens_per_sec"] / out["vram_used_mb_peak"]
    out["tokens_per_watt"] = out["tokens_per_sec"] / out["power_w_avg"]

    return out


samples = add_sample_metrics(samples)
samples.head()


,repo_id,served_model_id,model,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_percent_avg,gpu_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,backend,latency_s,ttft_s,ttft_share_of_latency,tokens_per_ram,tokens_per_vram,tokens_per_watt
0,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C,1,79.3070,133.0174,18.6184,4.6000,6.3,9370.37,895.34,68.6250,68.625,3.0,3.0,5254.0,47.9233,48.40,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.133017,0.079307,0.596215,0.020795,0.003544,0.388504
1,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,67.7943,122.0332,18.4369,6.7000,7.3,9375.46,895.34,68.6250,68.625,3.0,3.0,5254.0,49.9367,50.54,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.122033,0.067794,0.555540,0.020592,0.003509,0.369205
2,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,67.4295,122.5596,18.1389,5.6000,7.3,9376.24,895.34,68.5833,68.625,3.0,3.0,5254.0,51.2400,52.44,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.122560,0.067430,0.550177,0.020259,0.003452,0.353999
3,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,C,False,C,1,66.5789,121.0383,18.3623,6.0000,7.3,9353.32,895.34,68.5833,68.625,3.0,3.0,5254.0,52.9467,53.94,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.121038,0.066579,0.550065,0.020509,0.003495,0.346807
4,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,C,False,C,1,67.9756,122.0342,18.4985,5.5667,7.3,9364.48,895.34,68.6667,68.750,3.0,3.0,5254.0,54.7267,55.12,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.122034,0.067976,0.557021,0.020661,0.003521,0.338016


 Split scored and unscored datasets

In [8]:
scored_datasets = ["mmlu", "hellaswag", "gsm8k"]
summary_scored = summary[summary["dataset"].isin(scored_datasets)].copy()
summary_dolly = summary[summary["dataset"] == "dolly"].copy()

samples_scored = samples[samples["dataset"].isin(scored_datasets)].copy()
samples_dolly = samples[samples["dataset"] == "dolly"].copy()

print("summary_scored:", summary_scored.shape)
print("summary_dolly:", summary_dolly.shape)
print("samples_scored:", samples_scored.shape)
print("samples_dolly:", samples_dolly.shape)


summary_scored: (15, 46)
summary_dolly: (5, 46)
samples_scored: (15000, 41)
samples_dolly: (500, 41)


 Model x dataset summary

In [9]:
model_dataset_summary = (
    summary.groupby(["backend", "model", "dataset"], dropna=False)
    .agg(
        n_runs=("run_id", "nunique"),
        accuracy=("accuracy", "mean"),
        ttft=("ttft", "mean"),
        latency=("latency", "mean"),
        tps=("tps", "mean"),
        sustained_tps=("sustained_tps", "mean"),
        cpu=("cpu", "mean"),
        ram=("ram", "mean"),
        gpu=("gpu", "mean"),
        vram=("vram", "mean"),
        power=("power", "mean"),
        tps_per_vram=("tps_per_vram", "mean"),
        tps_per_watt=("tps_per_watt", "mean"),
    )
    .reset_index()
    .sort_values(["dataset", "model"])
)

display(model_dataset_summary)


,backend,model,dataset,n_runs,accuracy,ttft,latency,tps,sustained_tps,cpu,ram,gpu,vram,power,tps_per_vram,tps_per_watt
0,vllm,llama3_2_1b,dolly,1,NaN,73.7217,471.5200,109.4824,83.8610,7.9899,899.69,66.0237,5158.0,158.7051,0.021226,0.689848
4,vllm,ministral3_3b_awq,dolly,1,NaN,86.5122,582.3180,86.0036,67.4267,8.0497,909.82,34.0720,5165.0,170.7876,0.016651,0.503571
8,vllm,phi4mini_gptq_4bit,dolly,1,NaN,92.8500,634.2238,86.4342,67.7461,7.3774,881.47,75.1083,5930.0,169.1363,0.014576,0.511033
12,vllm,qwen2_5_1_5b,dolly,1,NaN,87.3424,612.8584,77.1278,61.9599,10.1223,823.19,56.1125,5404.0,154.1003,0.014272,0.500504
16,vllm,qwen3_4b_awq,dolly,1,NaN,100.5900,632.8397,85.4784,65.8724,7.4515,849.93,62.6153,5254.0,170.2425,0.016269,0.502098
1,vllm,llama3_2_1b,gsm8k,1,0.060,67.0925,138.3586,53.9784,23.6684,6.8413,899.57,44.0157,5370.0,103.1411,0.010052,0.523345
5,vllm,ministral3_3b_awq,gsm8k,1,0.050,70.9072,158.3890,65.6813,31.3323,7.2185,909.67,47.2187,5146.0,121.0202,0.012764,0.542730
9,vllm,phi4mini_gptq_4bit,gsm8k,1,0.190,71.1495,125.9490,20.9970,6.4244,6.4299,1030.05,11.9250,5917.0,66.5263,0.003549,0.315620
13,vllm,qwen2_5_1_5b,gsm8k,1,0.030,74.5962,154.9772,43.2927,21.7171,7.4701,823.03,32.7747,5406.0,102.1230,0.008008,0.423927
17,vllm,qwen3_4b_awq,gsm8k,1,0.050,87.9501,204.7330,84.7904,35.5647,6.7167,849.77,45.9030,5234.0,132.7710,0.016200,0.638621


 Scored-task overview per model

In [10]:
scored_model_overview = (
    summary_scored.groupby(["backend", "model"], dropna=False)
    .agg(
        n_runs=("run_id", "nunique"),
        n_datasets=("dataset", "nunique"),
        avg_accuracy=("accuracy", "mean"),
        avg_ttft_ms=("ttft", "mean"),
        avg_latency_ms=("latency", "mean"),
        avg_tps=("tps", "mean"),
        avg_sustained_tps=("sustained_tps", "mean"),
        avg_cpu=("cpu", "mean"),
        avg_ram_mb=("ram", "mean"),
        avg_gpu=("gpu", "mean"),
        avg_vram_mb=("vram", "mean"),
        avg_power_w=("power", "mean"),
        avg_tps_per_vram=("tps_per_vram", "mean"),
        avg_tps_per_watt=("tps_per_watt", "mean"),
        avg_accuracy_per_latency=("accuracy_per_latency", "mean"),
        avg_accuracy_per_vram=("accuracy_per_vram", "mean"),
        avg_accuracy_per_watt=("accuracy_per_watt", "mean"),
    )
    .reset_index()
)

display(scored_model_overview.sort_values("avg_accuracy", ascending=False))


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
4,vllm,qwen3_4b_awq,1,3,0.478000,99.074733,182.429967,39.535233,15.010633,6.025633,848.470000,32.574467,5283.000000,98.690900,0.007520,0.351571,0.002791,0.000090,0.005783
2,vllm,phi4mini_gptq_4bit,1,3,0.420000,72.685767,128.036433,39.943633,12.119133,6.530667,1069.146667,14.282533,5929.333333,69.607567,0.006734,0.566577,0.003269,0.000071,0.005978
1,vllm,ministral3_3b_awq,1,3,0.380000,82.970433,148.970300,33.962533,13.812333,6.365133,908.360000,34.762467,5231.666667,87.474600,0.006544,0.351647,0.002648,0.000072,0.005291
3,vllm,qwen2_5_1_5b,1,3,0.266667,73.553633,137.434067,26.539100,10.924767,6.679633,821.740000,19.106767,5408.333333,74.374267,0.004908,0.341456,0.002061,0.000049,0.004319
0,vllm,llama3_2_1b,1,3,0.246667,67.851067,127.813933,30.272567,11.680833,6.266567,898.283333,18.261133,5402.000000,73.187233,0.005617,0.385404,0.001994,0.000046,0.004089


 Dolly / generation overview per model

In [11]:
dolly_model_overview = (
    summary_dolly.groupby(["backend", "model"], dropna=False)
    .agg(
        n_runs=("run_id", "nunique"),
        avg_ttft_ms=("ttft", "mean"),
        avg_latency_ms=("latency", "mean"),
        avg_tps=("tps", "mean"),
        avg_sustained_tps=("sustained_tps", "mean"),
        avg_cpu=("cpu", "mean"),
        avg_ram_mb=("ram", "mean"),
        avg_gpu=("gpu", "mean"),
        avg_vram_mb=("vram", "mean"),
        avg_power_w=("power", "mean"),
        avg_tps_per_vram=("tps_per_vram", "mean"),
        avg_tps_per_watt=("tps_per_watt", "mean"),
    )
    .reset_index()
)

display(dolly_model_overview.sort_values("avg_tps", ascending=False))


,backend,model,n_runs,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt
0,vllm,llama3_2_1b,1,73.7217,471.5200,109.4824,83.8610,7.9899,899.69,66.0237,5158.0,158.7051,0.021226,0.689848
2,vllm,phi4mini_gptq_4bit,1,92.8500,634.2238,86.4342,67.7461,7.3774,881.47,75.1083,5930.0,169.1363,0.014576,0.511033
1,vllm,ministral3_3b_awq,1,86.5122,582.3180,86.0036,67.4267,8.0497,909.82,34.0720,5165.0,170.7876,0.016651,0.503571
4,vllm,qwen3_4b_awq,1,100.5900,632.8397,85.4784,65.8724,7.4515,849.93,62.6153,5254.0,170.2425,0.016269,0.502098
3,vllm,qwen2_5_1_5b,1,87.3424,612.8584,77.1278,61.9599,10.1223,823.19,56.1125,5404.0,154.1003,0.014272,0.500504


 Run-level stability from samples

In [12]:
run_level_from_samples = (
    samples.groupby(["backend", "model", "run_id", "dataset", "run_idx"], dropna=False)
    .agg(
        n_samples=("sample_idx", "count"),
        mean_output_tokens=("output_tokens", "mean"),
        mean_ttft_ms=("ttft_ms", "mean"),
        mean_latency_ms=("latency_ms", "mean"),
        mean_tokens_per_sec=("tokens_per_sec", "mean"),
        mean_cpu=("cpu_percent_avg", "mean"),
        mean_ram_mb=("process_ram_mb_peak", "mean"),
        mean_gpu=("gpu_percent_avg", "mean"),
        mean_vram_mb=("vram_used_mb_peak", "mean"),
        mean_power_w=("power_w_avg", "mean"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
)

display(run_level_from_samples.head(20))


,backend,model,run_id,dataset,run_idx,n_samples,mean_output_tokens,mean_ttft_ms,mean_latency_ms,mean_tokens_per_sec,mean_cpu,mean_ram_mb,mean_gpu,mean_vram_mb,mean_power_w,accuracy
0,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,1,10,43.90,70.557290,482.685460,108.838610,8.032000,899.6200,76.000000,5158.00,155.043530,NaN
1,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,2,10,55.80,67.894330,558.230070,113.800040,8.132000,899.6200,66.680000,5158.00,165.732600,NaN
2,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,3,10,41.90,87.125090,454.625320,112.113470,7.861170,899.6200,59.226670,5158.00,159.519860,NaN
3,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,4,10,41.10,71.292080,429.690790,112.120520,7.813660,899.6400,21.850000,5158.00,153.278880,NaN
4,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,5,10,38.60,72.989440,411.017760,107.789970,7.931160,899.6500,83.333330,5158.00,155.463160,NaN
5,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,6,10,36.70,77.263240,428.166290,105.438280,7.770170,899.6600,100.000000,5158.00,156.391790,NaN
6,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,7,10,44.40,70.061180,469.186790,108.463460,8.435000,899.6600,84.900000,5158.00,157.475910,NaN
7,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,8,10,37.80,68.961030,415.847310,105.838820,7.814160,899.6800,64.266670,5158.00,156.331450,NaN
8,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,9,10,51.70,71.883010,506.900840,117.508450,8.151170,899.6800,7.980000,5158.00,162.604020,NaN
9,vllm,llama3_2_1b,2026-04-07_13-35-14,dolly,10,10,49.30,79.190700,558.849030,102.912770,7.958500,899.6880,96.000000,5158.00,165.209750,NaN


 Incorrect scored samples

In [13]:
incorrect_samples = samples_scored[samples_scored["correct"] == False].copy()

cols_to_show = [
    "model", "run_id", "dataset", "run_idx", "sample_idx",
    "question", "gold_answer", "parsed_prediction",
    "raw_output", "ttft_ms", "latency_ms", "tokens_per_sec",
    "gpu_percent_avg", "vram_used_mb_peak", "power_w_avg",
]
cols_to_show = [c for c in cols_to_show if c in incorrect_samples.columns]

display(incorrect_samples[cols_to_show].head(50))


,model,run_id,dataset,run_idx,sample_idx,question,gold_answer,parsed_prediction,raw_output,ttft_ms,latency_ms,tokens_per_sec,gpu_percent_avg,vram_used_mb_peak,power_w_avg
2,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,1.0,C,C,67.4295,122.5596,18.1389,3.0,5254.0,51.2400
3,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,4,At which of the following locations does bile ...,1.0,C,C,66.5789,121.0383,18.3623,3.0,5254.0,52.9467
4,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,5,"Let A be a finite set with m elements, and let...",0.0,C,C,67.9756,122.0342,18.4985,3.0,5254.0,54.7267
5,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,6,An employee of large manufacturing firm finds ...,1.0,D,D,67.5544,121.2762,18.6144,3.0,5254.0,56.6900
6,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,7,A state imposes a graduated income tax upon ne...,2.0,A,A,68.5019,123.2268,18.2732,3.0,5248.0,57.5167
8,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,9,What is the meaning of collective security?,2.0,A,A,68.1858,122.1616,18.5268,3.0,5243.0,58.2700
10,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,11,Let T be a depth-first search tree of a connec...,3.0,C,C,68.5249,122.1281,18.6556,3.0,5242.0,57.8033
12,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,13,Anscombe criticizes as absurd Kant’s idea of:,3.0,C,C,67.9626,121.9028,18.5391,3.0,5239.0,58.5633
13,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,14,A ________ campaign occurs when people from tw...,1.0,C,C,67.3250,121.5651,18.4366,3.0,5239.0,59.0700
14,llama3_2_1b,2026-04-07_13-35-14,mmlu,1,15,For which of these two scenarios does the main...,1.0,D,D,68.5530,122.6176,18.4964,3.0,5239.0,58.1167


 Sample-level aggregate summary

In [14]:
sample_aggregate_summary = (
    samples.groupby(["backend", "model", "dataset"], dropna=False)
    .agg(
        n_rows=("sample_idx", "count"),
        mean_output_tokens=("output_tokens", "mean"),
        mean_ttft_ms=("ttft_ms", "mean"),
        mean_latency_ms=("latency_ms", "mean"),
        mean_tokens_per_sec=("tokens_per_sec", "mean"),
        mean_cpu=("cpu_percent_avg", "mean"),
        mean_ram_mb=("process_ram_mb_peak", "mean"),
        mean_gpu=("gpu_percent_avg", "mean"),
        mean_vram_mb=("vram_used_mb_peak", "mean"),
        mean_power_w=("power_w_avg", "mean"),
        mean_tokens_per_vram=("tokens_per_vram", "mean"),
        mean_tokens_per_watt=("tokens_per_watt", "mean"),
        pct_correct=("correct", "mean"),
    )
    .reset_index()
    .sort_values(["dataset", "model"])
)

display(sample_aggregate_summary)


,backend,model,dataset,n_rows,mean_output_tokens,mean_ttft_ms,mean_latency_ms,mean_tokens_per_sec,mean_cpu,mean_ram_mb,mean_gpu,mean_vram_mb,mean_power_w,mean_tokens_per_vram,mean_tokens_per_watt,pct_correct
0,vllm,llama3_2_1b,dolly,100,44.120,73.721739,471.519966,109.482439,7.989899,899.65180,66.023667,5158.000,158.705095,0.021226,0.729053,NaN
4,vllm,ministral3_3b_awq,dolly,100,42.920,86.512189,582.317982,86.003644,8.049717,909.76290,34.072000,5146.600,170.787595,0.016711,0.528480,NaN
8,vllm,phi4mini_gptq_4bit,dolly,100,47.580,92.849954,634.223796,86.434243,7.377448,881.41300,75.108333,5925.600,169.136280,0.014587,0.534821,NaN
12,vllm,qwen2_5_1_5b,dolly,100,41.890,87.342369,612.858379,77.127822,10.122334,823.12580,56.112499,5404.000,154.100323,0.014272,0.526572,NaN
16,vllm,qwen3_4b_awq,dolly,100,46.420,100.590015,632.839732,85.478390,7.451534,849.87560,62.615333,5234.960,170.242487,0.016328,0.526895,NaN
1,vllm,llama3_2_1b,gsm8k,1000,4.520,67.092467,138.358580,53.978440,6.841266,898.93560,44.015667,5196.816,103.141072,0.010388,0.526473,0.06
5,vllm,ministral3_3b_awq,gsm8k,1000,6.647,70.907210,158.388997,65.681343,7.218499,909.02893,47.218667,5145.522,121.020198,0.012765,0.547118,0.05
9,vllm,phi4mini_gptq_4bit,gsm8k,1000,1.150,71.149501,125.949022,20.997007,6.429934,947.19797,11.925000,5915.054,66.526298,0.003550,0.316840,0.19
13,vllm,qwen2_5_1_5b,gsm8k,1000,4.750,74.596179,154.977242,43.292655,7.470066,822.36721,32.774667,5404.578,102.123004,0.008010,0.425526,0.03
17,vllm,qwen3_4b_awq,gsm8k,1000,10.103,87.950051,204.732966,84.790417,6.716699,849.11962,45.902999,5234.000,132.771033,0.016200,0.643677,0.05


 Rankings

In [15]:
ranking_accuracy = scored_model_overview.sort_values(
    ["avg_accuracy", "avg_tps_per_watt", "avg_tps_per_vram"],
    ascending=[False, False, False]
).reset_index(drop=True)

ranking_efficiency = scored_model_overview.sort_values(
    ["avg_tps_per_watt", "avg_accuracy", "avg_tps_per_vram"],
    ascending=[False, False, False]
).reset_index(drop=True)

display(ranking_accuracy)
display(ranking_efficiency)


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,qwen3_4b_awq,1,3,0.478000,99.074733,182.429967,39.535233,15.010633,6.025633,848.470000,32.574467,5283.000000,98.690900,0.007520,0.351571,0.002791,0.000090,0.005783
1,vllm,phi4mini_gptq_4bit,1,3,0.420000,72.685767,128.036433,39.943633,12.119133,6.530667,1069.146667,14.282533,5929.333333,69.607567,0.006734,0.566577,0.003269,0.000071,0.005978
2,vllm,ministral3_3b_awq,1,3,0.380000,82.970433,148.970300,33.962533,13.812333,6.365133,908.360000,34.762467,5231.666667,87.474600,0.006544,0.351647,0.002648,0.000072,0.005291
3,vllm,qwen2_5_1_5b,1,3,0.266667,73.553633,137.434067,26.539100,10.924767,6.679633,821.740000,19.106767,5408.333333,74.374267,0.004908,0.341456,0.002061,0.000049,0.004319
4,vllm,llama3_2_1b,1,3,0.246667,67.851067,127.813933,30.272567,11.680833,6.266567,898.283333,18.261133,5402.000000,73.187233,0.005617,0.385404,0.001994,0.000046,0.004089


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,phi4mini_gptq_4bit,1,3,0.420000,72.685767,128.036433,39.943633,12.119133,6.530667,1069.146667,14.282533,5929.333333,69.607567,0.006734,0.566577,0.003269,0.000071,0.005978
1,vllm,llama3_2_1b,1,3,0.246667,67.851067,127.813933,30.272567,11.680833,6.266567,898.283333,18.261133,5402.000000,73.187233,0.005617,0.385404,0.001994,0.000046,0.004089
2,vllm,ministral3_3b_awq,1,3,0.380000,82.970433,148.970300,33.962533,13.812333,6.365133,908.360000,34.762467,5231.666667,87.474600,0.006544,0.351647,0.002648,0.000072,0.005291
3,vllm,qwen3_4b_awq,1,3,0.478000,99.074733,182.429967,39.535233,15.010633,6.025633,848.470000,32.574467,5283.000000,98.690900,0.007520,0.351571,0.002791,0.000090,0.005783
4,vllm,qwen2_5_1_5b,1,3,0.266667,73.553633,137.434067,26.539100,10.924767,6.679633,821.740000,19.106767,5408.333333,74.374267,0.004908,0.341456,0.002061,0.000049,0.004319


 Output paths

In [16]:
summary_normalized_path = PROCESSED_DIR / "summary_vllm.csv"
summary_scored_path = PROCESSED_DIR / "summary_scored_vllm.csv"
summary_dolly_path = PROCESSED_DIR / "summary_dolly_vllm.csv"

samples_all_path = PROCESSED_DIR / "samples_vllm.csv"
samples_scored_path = PROCESSED_DIR / "samples_scored_vllm.csv"
samples_dolly_path = PROCESSED_DIR / "samples_dolly_vllm.csv"

model_dataset_summary_path = PROCESSED_DIR / "model_dataset_summary_vllm.csv"
scored_model_overview_path = PROCESSED_DIR / "scored_model_overview_vllm.csv"
dolly_model_overview_path = PROCESSED_DIR / "dolly_model_overview_vllm.csv"
run_level_from_samples_path = PROCESSED_DIR / "run_level_from_samples_vllm.csv"
sample_aggregate_summary_path = PROCESSED_DIR / "sample_aggregate_summary_vllm.csv"
incorrect_samples_path = PROCESSED_DIR / "incorrect_samples_vllm.csv"
ranking_accuracy_path = PROCESSED_DIR / "ranking_accuracy_vllm.csv"
ranking_efficiency_path = PROCESSED_DIR / "ranking_efficiency_vllm.csv"


 Save processed outputs

In [17]:
summary.to_csv(summary_normalized_path, index=False)
summary_scored.to_csv(summary_scored_path, index=False)
summary_dolly.to_csv(summary_dolly_path, index=False)

samples.to_csv(samples_all_path, index=False)
samples_scored.to_csv(samples_scored_path, index=False)
samples_dolly.to_csv(samples_dolly_path, index=False)

model_dataset_summary.to_csv(model_dataset_summary_path, index=False)
scored_model_overview.to_csv(scored_model_overview_path, index=False)
dolly_model_overview.to_csv(dolly_model_overview_path, index=False)
run_level_from_samples.to_csv(run_level_from_samples_path, index=False)
sample_aggregate_summary.to_csv(sample_aggregate_summary_path, index=False)
incorrect_samples.to_csv(incorrect_samples_path, index=False)
ranking_accuracy.to_csv(ranking_accuracy_path, index=False)
ranking_efficiency.to_csv(ranking_efficiency_path, index=False)

print("Saved:")
for p in [
    summary_normalized_path,
    summary_scored_path,
    summary_dolly_path,
    samples_all_path,
    samples_scored_path,
    samples_dolly_path,
    model_dataset_summary_path,
    scored_model_overview_path,
    dolly_model_overview_path,
    run_level_from_samples_path,
    sample_aggregate_summary_path,
    incorrect_samples_path,
    ranking_accuracy_path,
    ranking_efficiency_path,
]:
    print("-", p.name)


Saved:
- summary_vllm.csv
- summary_scored_vllm.csv
- summary_dolly_vllm.csv
- samples_vllm.csv
- samples_scored_vllm.csv
- samples_dolly_vllm.csv
- model_dataset_summary_vllm.csv
- scored_model_overview_vllm.csv
- dolly_model_overview_vllm.csv
- run_level_from_samples_vllm.csv
- sample_aggregate_summary_vllm.csv
- incorrect_samples_vllm.csv
- ranking_accuracy_vllm.csv
- ranking_efficiency_vllm.csv


 Final inspection

In [18]:
display(summary.head())
display(samples.head())
display(ranking_accuracy.head())
display(ranking_efficiency.head())

,backend,run_id,dataset,model_alias_dir,source_path,model,repo_id,served_model_id,accuracy,accuracy_sd,ttft,ttft_sd,latency,latency_sd,tps,tps_sd,cpu,cpu_sd,cpu_peak,ram_used_mb_peak,ram,temp_c_avg,temp_c_avg_sd,temp_c_peak,gpu,gpu_sd,gpu_peak,vram,power,power_sd,power_peak,sustained_tps,sustained_tps_sd,sustained_sps,sustained_sps_sd,latency_s,ttft_s,ttft_share_of_latency,tps_per_ram,tps_per_vram,tps_per_watt,sustained_tps_per_vram,accuracy_per_latency,accuracy_per_ram,accuracy_per_vram,accuracy_per_watt
0,vllm,2026-04-07_13-35-14,mmlu,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.44,0.0,68.2622,0.3665,122.6156,0.7112,18.4056,0.1139,6.0300,0.2637,39.2,9376.24,896.96,68.5707,0.6288,71.500,8.2657,9.4386,22.0,5436.0,58.1622,0.6932,61.44,5.6841,0.0316,5.6841,0.0316,0.122616,0.068262,0.556717,0.020520,0.003386,0.316453,0.001046,0.003588,0.000491,0.000081,0.007565
1,vllm,2026-04-07_13-35-14,hellaswag,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.24,0.0,68.1985,0.1777,122.4676,0.4120,18.4337,0.0793,5.9284,0.1299,43.4,9272.16,898.32,67.6965,0.2944,68.625,2.5020,6.6964,22.0,5400.0,58.2584,0.0683,60.20,5.6900,0.0191,5.6900,0.0191,0.122468,0.068198,0.556870,0.020520,0.003414,0.316413,0.001054,0.001960,0.000267,0.000044,0.004120
2,vllm,2026-04-07_13-35-14,gsm8k,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,0.06,0.0,67.0925,0.2472,138.3586,0.3620,53.9784,0.0768,6.8413,0.0165,23.6,9387.93,899.57,71.7561,0.2473,73.250,44.0157,15.2163,100.0,5370.0,103.1411,1.4601,128.19,23.6684,0.0713,5.2364,0.0158,0.138359,0.067092,0.484917,0.060005,0.010052,0.523345,0.004408,0.000434,0.000067,0.000011,0.000582
3,vllm,2026-04-07_13-35-14,dolly,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,NaN,NaN,73.7217,5.8875,471.5200,54.9763,109.4824,4.3976,7.9899,0.2048,22.2,9301.80,899.69,74.9008,0.1878,76.250,66.0237,30.1451,100.0,5158.0,158.7051,4.3896,187.04,83.8610,4.9172,1.9239,0.1934,0.471520,0.073722,0.156349,0.121689,0.021226,0.689848,0.016258,NaN,NaN,NaN,NaN
4,vllm,2026-04-07_13-56-00,mmlu,ministral3_3b_awq,/home/stephane/resource-efficiency-benchmark/r...,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,0.64,0.0,82.6019,0.9471,137.8528,0.9156,18.1055,0.0572,5.9592,0.0430,23.9,10783.76,907.05,70.0784,0.2781,73.250,28.6720,8.2209,100.0,5396.0,69.6547,1.8607,102.96,5.2170,0.0256,5.2170,0.0256,0.137853,0.082602,0.599204,0.019961,0.003355,0.259932,0.000967,0.004643,0.000706,0.000119,0.009188


,repo_id,served_model_id,model,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_percent_avg,gpu_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,backend,latency_s,ttft_s,ttft_share_of_latency,tokens_per_ram,tokens_per_vram,tokens_per_watt
0,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C,1,79.3070,133.0174,18.6184,4.6000,6.3,9370.37,895.34,68.6250,68.625,3.0,3.0,5254.0,47.9233,48.40,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.133017,0.079307,0.596215,0.020795,0.003544,0.388504
1,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,67.7943,122.0332,18.4369,6.7000,7.3,9375.46,895.34,68.6250,68.625,3.0,3.0,5254.0,49.9367,50.54,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.122033,0.067794,0.555540,0.020592,0.003509,0.369205
2,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,67.4295,122.5596,18.1389,5.6000,7.3,9376.24,895.34,68.5833,68.625,3.0,3.0,5254.0,51.2400,52.44,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.122560,0.067430,0.550177,0.020259,0.003452,0.353999
3,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,C,False,C,1,66.5789,121.0383,18.3623,6.0000,7.3,9353.32,895.34,68.5833,68.625,3.0,3.0,5254.0,52.9467,53.94,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.121038,0.066579,0.550065,0.020509,0.003495,0.346807
4,unsloth/Llama-3.2-1B-Instruct,unsloth/Llama-3.2-1B-Instruct,llama3_2_1b,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,C,False,C,1,67.9756,122.0342,18.4985,5.5667,7.3,9364.48,895.34,68.6667,68.750,3.0,3.0,5254.0,54.7267,55.12,llama3_2_1b,2026-04-07_13-35-14,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.122034,0.067976,0.557021,0.020661,0.003521,0.338016


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,qwen3_4b_awq,1,3,0.478000,99.074733,182.429967,39.535233,15.010633,6.025633,848.470000,32.574467,5283.000000,98.690900,0.007520,0.351571,0.002791,0.000090,0.005783
1,vllm,phi4mini_gptq_4bit,1,3,0.420000,72.685767,128.036433,39.943633,12.119133,6.530667,1069.146667,14.282533,5929.333333,69.607567,0.006734,0.566577,0.003269,0.000071,0.005978
2,vllm,ministral3_3b_awq,1,3,0.380000,82.970433,148.970300,33.962533,13.812333,6.365133,908.360000,34.762467,5231.666667,87.474600,0.006544,0.351647,0.002648,0.000072,0.005291
3,vllm,qwen2_5_1_5b,1,3,0.266667,73.553633,137.434067,26.539100,10.924767,6.679633,821.740000,19.106767,5408.333333,74.374267,0.004908,0.341456,0.002061,0.000049,0.004319
4,vllm,llama3_2_1b,1,3,0.246667,67.851067,127.813933,30.272567,11.680833,6.266567,898.283333,18.261133,5402.000000,73.187233,0.005617,0.385404,0.001994,0.000046,0.004089


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,phi4mini_gptq_4bit,1,3,0.420000,72.685767,128.036433,39.943633,12.119133,6.530667,1069.146667,14.282533,5929.333333,69.607567,0.006734,0.566577,0.003269,0.000071,0.005978
1,vllm,llama3_2_1b,1,3,0.246667,67.851067,127.813933,30.272567,11.680833,6.266567,898.283333,18.261133,5402.000000,73.187233,0.005617,0.385404,0.001994,0.000046,0.004089
2,vllm,ministral3_3b_awq,1,3,0.380000,82.970433,148.970300,33.962533,13.812333,6.365133,908.360000,34.762467,5231.666667,87.474600,0.006544,0.351647,0.002648,0.000072,0.005291
3,vllm,qwen3_4b_awq,1,3,0.478000,99.074733,182.429967,39.535233,15.010633,6.025633,848.470000,32.574467,5283.000000,98.690900,0.007520,0.351571,0.002791,0.000090,0.005783
4,vllm,qwen2_5_1_5b,1,3,0.266667,73.553633,137.434067,26.539100,10.924767,6.679633,821.740000,19.106767,5408.333333,74.374267,0.004908,0.341456,0.002061,0.000049,0.004319
